In [ ]:
from __future__ import print_function, division

import os
import re
import glob

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors, gridspec
import matplotlib.patheffects as pe
from matplotlib.ticker import MultipleLocator

%matplotlib inline

try:
    from IPython.display import Image as IPyImage, display
except ImportError:
    IPyImage = display = None

# parametri e percorsi configurabili (caricati dinamicamente da config.py)
import sys
from pathlib import Path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
import config

INPUT_DIR = str(config.DATA_DIR)
OUTPUT_FILE = str(config.OUTPUT_DIR / "analysis")
FILE_PATTERN = config.FILE_PATTERN
COMMENT_CHAR = config.COMMENT_CHAR
TEMP_PATTERN = config.TEMP_PATTERN
TEMP_UNIT = config.TEMP_UNIT
TWO_THETA_MIN = config.TWO_THETA_MIN
TWO_THETA_MAX = config.TWO_THETA_MAX
TWO_THETA_CROP_MIN = config.TWO_THETA_CROP_MIN

PLOT_2THETA_MIN = None
PLOT_2THETA_MAX = None
WATERFALL_OFFSET = 0.0
SHOW_PEAK_TRACK = True
CMAP_NAME = "viridis"

# due gif: da T ambiente a SPLIT_TEMP e da SPLIT_TEMP alla fine
MAKE_GIF = True
SPLIT_TEMP = 500
GIF_CUMULATIVE = True
GIF_STEP = 3
GIF_FPS = 10
GIF_DPI = 120
GIF_HOLD_LAST_S = 1.5
GIF_SHARED_COLORS = False
GIF_LOOP = False                 # True = la gif si ripete
MAKE_MP4 = False
MAKE_PPTX = False

# riferimenti ICDD sotto il grafico
SHOW_REFS = True
LAMBDA_CU_KA1 = 1.540598
LABEL_MIN_I = 8
MIN_LABEL_SEP = 0.02
REF_T_C = 25.0

# etichette sui picchi
SHOW_PEAK_LABELS = True
LABEL_FONTSIZE = 12
LABELS_ON_ALL_SHOWN = False

# (indice, 2theta min, 2theta max)
MXENE_LABELS = [("002", 9.0, 11.0), ("100", 31.0, 33.0), ("103", 37.0, 39.0)]
MXENE_LABEL_COLOR = "#8c1515"
MXENE_PRESENCE_SIGMA = 0.0

# (indice, centro, +/- finestra)
RUTILE_LABELS = [("101", 36.085, 0.6), ("002", 62.740, 2.0),("111", 41.225, 0.3)]
RUTILE_LABEL_COLOR = "#0072B2"
RUTILE_PRESENCE_SIGMA = 3.0

# (2theta, intensita', h, k, l)
PHASES = [
    {"name": "Ti2AlC0.5N0.5 (PDF 00-029-0095)", "color": "#d62728", "alpha": 0.0, "peaks": [
        (13.008, 39, 0, 0, 2), (34.021, 19, 1, 0, 0), (34.673, 6, 1, 0, 1),
        (39.545, 100, 1, 0, 3), (39.727, 18, 0, 0, 6), (43.428, 3, 1, 0, 4),
        (53.287, 14, 1, 0, 6), (60.897, 14, 1, 1, 0), (71.626, 2, 2, 0, 0),
        (71.992, 9, 1, 0, 9), (75.204, 11, 1, 1, 6)]},
    {"name": "TiO2 (PDF 00-021-1276)", "color": "#0072B2", "alpha": 0.0, "peaks": [
        (27.446, 100, 1, 1, 0), (36.085, 50, 1, 0, 1), (39.187, 8, 2, 0, 0),
        (41.225, 25, 1, 1, 1), (44.050, 10, 2, 1, 0), (54.322, 60, 2, 1, 1),
        (56.640, 20, 2, 2, 0), (62.740, 10, 0, 0, 2), (64.038, 10, 3, 1, 0),
        (65.478, 2, 2, 2, 1), (69.008, 20, 3, 0, 1), (69.788, 12, 1, 1, 2),
        (72.408, 2, 3, 1, 1), (74.409, 1, 3, 2, 0), (76.508, 4, 2, 0, 2),
        (79.819, 2, 2, 1, 2)]},
]


def extract_temp(filename):
    match = TEMP_PATTERN.search(filename)
    if match is None:
        print("[{0}] warning: could not extract temperature from filename".format(filename))
        return np.nan
    return float(match.group(1))


def read_data(path):
    try:
        data = np.loadtxt(path, comments=COMMENT_CHAR, ndmin=2)
    except ValueError:
        data = np.loadtxt(path, comments=COMMENT_CHAR, delimiter=",", ndmin=2)
    if data.shape[1] < 2:
        raise ValueError("servono almeno 2 colonne (2theta, intensita')")
    return data


def peak_in_range(two_theta, intensity, lo, hi, name=""):
    mask = (two_theta >= lo) & (two_theta <= hi)
    if not mask.any():
        print("[{0}] warning: no points found in range [{1}, {2}] deg".format(name, lo, hi))
        return np.nan
    return two_theta[mask][np.argmax(intensity[mask])]


def write_gif(path, arrays, fps):
    try:
        import PIL
        from PIL import Image
        try:
            ver = tuple(int(x) for x in PIL.__version__.split(".")[:2])
        except Exception:
            ver = (99, 0)
    except ImportError:
        Image = None
        ver = (0, 0)

    if Image is not None and ver >= (3, 4):
        imgs = [Image.fromarray(a).convert("P", palette=Image.ADAPTIVE) for a in arrays]
        opts = dict(save_all=True, append_images=imgs[1:], duration=int(1000 / fps))
        if GIF_LOOP:
            opts["loop"] = 0
        imgs[0].save(path, **opts)
        return "Pillow"

    try:
        import imageio
    except ImportError:
        raise RuntimeError("per la GIF serve Pillow >= 3.4 oppure imageio: "
                           "esegui  pip install \"pillow<7\"  e riavvia il kernel")
    imageio.mimsave(path, arrays, duration=1.0 / fps)
    return "imageio"


def write_mp4(path, arrays, fps):
    import imageio
    even = []
    for a in arrays:
        h, w = a.shape[:2]
        even.append(a[:h - h % 2, :w - w % 2])
    imageio.mimsave(path, even, fps=fps, codec="libx264", macro_block_size=1)


def build_pptx(path, slides):
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from PIL import Image

    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank = prs.slide_layouts[6]
    max_w, max_h = int(Inches(12.3)), int(Inches(6.1))

    for title, img in slides:
        slide = prs.slides.add_slide(blank)
        tb = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(12.3), Inches(0.7))
        tb.text_frame.text = title
        tb.text_frame.paragraphs[0].runs[0].font.size = Pt(28)

        w_px, h_px = Image.open(img).size
        scale = min(max_w / float(w_px), max_h / float(h_px))
        w, h = int(w_px * scale), int(h_px * scale)
        left = int((prs.slide_width - w) / 2)
        slide.shapes.add_picture(img, left, int(Inches(1.05)), w, h)
    prs.save(path)


def shifted_two_theta(tt0, temp, alpha):
    tt0 = np.asarray(tt0, dtype=float)
    if not alpha:
        return tt0
    d0 = LAMBDA_CU_KA1 / (2.0 * np.sin(np.radians(tt0 / 2.0)))
    d = d0 * (1.0 + alpha * (temp - REF_T_C))
    return 2.0 * np.degrees(np.arcsin(LAMBDA_CU_KA1 / (2.0 * d)))


def hkl_label(h, k, l):
    idx = (h, k, l)
    sep = " " if max(idx) >= 10 else ""
    return "(" + sep.join(str(i) for i in idx) + ")"


def draw_refs(ax, lo, hi, temp):
    n = len(PHASES)
    ax.set_xlim(lo, hi)
    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.set_ylim(-0.08, float(n))
    ax.set_yticks([])
    ax.set_xlabel("2theta (deg)")
    for i, ph in enumerate(PHASES):
        base = float(n - 1 - i)
        ax.axhline(base, color="0.8", lw=0.6)
        tt = shifted_two_theta([pk[0] for pk in ph["peaks"]], temp, ph.get("alpha", 0.0))
        pairs = sorted(zip(tt, ph["peaks"]), key=lambda z: -z[1][1])
        placed = []
        for x, (tt0, inten, h, k, l) in pairs:
            if x < lo or x > hi:
                continue
            height = 0.12 + 0.33 * inten / 100.0
            ax.plot([x, x], [base, base + height], color=ph["color"], lw=1.4)
            if inten >= LABEL_MIN_I and all(abs(x - px) >= MIN_LABEL_SEP * (hi - lo) for px in placed):
                ax.text(x, base + height + 0.03, hkl_label(h, k, l), rotation=90,
                        ha="center", va="bottom", fontsize=6, color=ph["color"])
                placed.append(x)
        ax.text(hi, base + 0.98, ph["name"], ha="right", va="top", fontsize=7,
                color=ph["color"], fontweight="bold")


def window_peak(tt, inten, w_lo, w_hi, sigma=0.0):
    info = {"reason": "", "x": np.nan, "height": 0.0, "required": 0.0}
    idx = np.where((tt >= w_lo) & (tt <= w_hi))[0]
    if len(idx) < 3:
        info["reason"] = "nessun dato nella finestra"
        return None, info
    i_max = idx[np.argmax(inten[idx])]
    x_m, y_m = tt[i_max], inten[i_max]
    info["x"] = x_m
    if sigma > 0:
        width = w_hi - w_lo
        left = np.where((tt >= w_lo - width) & (tt < w_lo))[0]
        right = np.where((tt > w_hi) & (tt <= w_hi + width))[0]
        wide = np.where((tt >= w_lo - width) & (tt <= w_hi + width))[0]
        if i_max == idx[0] or i_max == idx[-1]:
            info["reason"] = "il massimo e' sul bordo della finestra (non c'e' un picco)"
            return None, info
        if len(left) >= 3 and len(right) >= 3:
            xl, yl = np.median(tt[left]), np.median(inten[left])
            xr, yr = np.median(tt[right]), np.median(inten[right])
            base = yl + (yr - yl) * (x_m - xl) / (xr - xl)
        else:
            base = np.median(inten[wide])
        d = np.diff(inten[wide])
        noise = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2.0)
        info["height"] = y_m - base
        info["required"] = sigma * noise
        if info["height"] <= 0 or info["height"] < info["required"]:
            info["reason"] = "troppo basso rispetto al rumore"
            return None, info
    return (x_m, y_m), info


def build_label_specs(with_rutile):
    specs = []
    for text, a, b in MXENE_LABELS:
        specs.append({"key": "MXene " + text, "text": text, "w_lo": a, "w_hi": b,
                      "color": MXENE_LABEL_COLOR, "sigma": MXENE_PRESENCE_SIGMA})
    if with_rutile:
        for text, c, w in RUTILE_LABELS:
            specs.append({"key": "Rutile " + text, "text": text, "w_lo": c - w, "w_hi": c + w,
                          "color": RUTILE_LABEL_COLOR, "sigma": RUTILE_PRESENCE_SIGMA})
    return specs


def make_gif(pats, gif_path, cmap, lo, hi, peak_by_name, tag, norm_global, label_specs=()):
    n = len(pats)
    temps_g = np.array([p[0] for p in pats])
    if GIF_SHARED_COLORS:
        norm_g = norm_global
    else:
        norm_g = colors.Normalize(vmin=temps_g.min(), vmax=temps_g.max())

    idxs = list(range(0, n, max(1, int(GIF_STEP))))
    if idxs[-1] != n - 1:
        idxs.append(n - 1)

    max_int, y_lo = 0.0, 0.0
    for temp, tt, inten, name in pats:
        m = (tt >= lo) & (tt <= hi)
        if m.any():
            max_int = max(max_int, inten[m].max())
            y_lo = min(y_lo, inten[m].min())
    step_g = WATERFALL_OFFSET * max_int
    y_hi = (max_int + step_g * (n - 1)) * (1.12 if SHOW_PEAK_LABELS else 1.05)

    srt = []
    for temp, tt, inten, name in pats:
        o = np.argsort(tt)
        srt.append((tt[o], inten[o]))

    n_ph = len(PHASES) if SHOW_REFS else 0
    if n_ph:
        fig_g = plt.figure(figsize=(8, 4.8 + 0.8 * n_ph), dpi=GIF_DPI)
        gs = gridspec.GridSpec(2, 1, height_ratios=[4.4, 0.8 * n_ph], hspace=0.06,
                               left=0.11, right=0.86, bottom=0.10, top=0.90)
        ax_g = fig_g.add_subplot(gs[0])
        ax_r = fig_g.add_subplot(gs[1])
    else:
        fig_g, ax_g = plt.subplots(figsize=(8, 5), dpi=GIF_DPI)
        fig_g.subplots_adjust(left=0.11, right=0.86, bottom=0.12, top=0.86)
        ax_r = None

    pos = ax_g.get_position()
    cax = fig_g.add_axes([0.89, pos.y0, 0.025, pos.height])
    sm_g = cm.ScalarMappable(norm=norm_g, cmap=cmap)
    sm_g.set_array([])
    fig_g.colorbar(sm_g, cax=cax).set_label("Temperature ({0})".format(TEMP_UNIT))
    t_marker = None
    shown_count = {}
    last_fail = {}

    frames = []
    for k in idxs:
        ax_g.clear()
        if ax_r is not None:
            ax_r.clear()
        shown = [j for j in idxs if j <= k] if GIF_CUMULATIVE else [k]
        for j in shown:
            temp_j, tt, inten, name = pats[j]
            m = (tt >= lo) & (tt <= hi)
            current = (j == k)
            ax_g.plot(tt[m], inten[m] + j * step_g, color=cmap(norm_g(temp_j)),
                      lw=2.0 if current else 0.8, alpha=1.0 if current else 0.6)

        temp_k = pats[k][0]
        label = "T = {0:.0f} {1}".format(temp_k, TEMP_UNIT)
        if t_marker is not None:
            t_marker.remove()
        t_marker = cax.axhline(temp_k, color="k", lw=2.0)
        pk = peak_by_name.get(pats[k][3], np.nan)
        if np.isfinite(pk) and lo <= pk <= hi:
            ax_g.axvline(pk, color="k", ls="--", lw=0.8)
            label += "   |   peak 2theta = {0:.3f}".format(pk)

        if label_specs:
            label_js = shown if LABELS_ON_ALL_SHOWN else [k]
            dy = 0.01 * (y_hi - y_lo)
            for j in label_js:
                tt_s, in_s = srt[j]
                for sp in label_specs:
                    found, info = window_peak(tt_s, in_s, sp["w_lo"], sp["w_hi"], sp["sigma"])
                    if found is None:
                        if j == k:
                            last_fail[sp["key"]] = info
                        continue
                    if j == k:
                        shown_count[sp["key"]] = shown_count.get(sp["key"], 0) + 1
                    ax_g.text(found[0], found[1] + j * step_g + dy, sp["text"],
                              ha="center", va="bottom", fontsize=LABEL_FONTSIZE,
                              fontweight="bold", color=sp["color"],
                              path_effects=[pe.withStroke(linewidth=3, foreground="white")])

        ax_g.set_xlim(lo, hi)
        ax_g.set_ylim(y_lo, y_hi)
        ax_g.set_ylabel("Intensity (a.u.)")
        ax_g.set_title(tag + "\n" + label, fontsize=10)
        if ax_r is not None:
            ax_g.tick_params(labelbottom=False)
            draw_refs(ax_r, lo, hi, temp_k)
        else:
            ax_g.set_xlabel("2theta (deg)")

        fig_g.canvas.draw()
        w, h = fig_g.canvas.get_width_height()
        rgba = np.frombuffer(fig_g.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, 4)
        frames.append(rgba[:, :, :3].copy())
    plt.close(fig_g)

    for sp in label_specs:
        if shown_count.get(sp["key"], 0) == 0:
            info = last_fail.get(sp["key"], {})
            line = "  ATTENZIONE: etichetta {0} (finestra {1:.2f}-{2:.2f}) mai mostrata in questa GIF".format(
                sp["key"], sp["w_lo"], sp["w_hi"])
            if info.get("reason"):
                line += ": " + info["reason"]
                if np.isfinite(info.get("x", np.nan)) and info.get("required", 0) > 0:
                    line += " (massimo a {0:.3f}, altezza {1:.3g} su soglia {2:.3g})".format(
                        info["x"], info["height"], info["required"])
            print(line)

    frames.extend([frames[-1]] * int(round(GIF_HOLD_LAST_S * GIF_FPS)))
    how = write_gif(gif_path, frames, GIF_FPS)
    return how, len(frames), len(idxs), frames


# lettura file e ricerca dei picchi
files = sorted(glob.glob(os.path.join(INPUT_DIR, FILE_PATTERN)))
if not files:
    print("No files matching {0} found in {1}".format(FILE_PATTERN, INPUT_DIR))

results = []
patterns = []
peak_by_name = {}
for f in files:
    name = os.path.basename(f)
    temp = extract_temp(name)
    try:
        data = read_data(f)
    except Exception as e:
        print("[{0}] ERROR: {1}".format(name, e))
        results.append((temp, np.nan))
        continue
    if TWO_THETA_CROP_MIN is not None:
        data = data[data[:, 0] >= TWO_THETA_CROP_MIN]
        if data.shape[0] == 0:
            print("[{0}] warning: no points left above {1} deg".format(name, TWO_THETA_CROP_MIN))
            results.append((temp, np.nan))
            continue
    tt, inten = data[:, 0], data[:, 1]
    peak = peak_in_range(tt, inten, TWO_THETA_MIN, TWO_THETA_MAX, name)
    results.append((temp, peak))
    patterns.append((temp, tt, inten, name))
    peak_by_name[name] = peak
    print("[{0}] T={1:g}, peak_position={2:.4f}".format(name, temp, peak))

# tabella
if results:
    if os.path.isdir(OUTPUT_FILE):
        raise IOError("OUTPUT_FILE e' una cartella, indica un nome di file: " + OUTPUT_FILE)
    out_dir = os.path.dirname(OUTPUT_FILE)
    if out_dir and not os.path.isdir(out_dir):
        os.makedirs(out_dir)

    table = np.array(results, dtype=float)
    table = table[np.argsort(table[:, 0], kind="mergesort")]

    header = (
        "# Peak position (max intensity) in range [{0}, {1}] deg vs temperature\n"
        "# columns: temperature  peak_position_2theta"
    ).format(TWO_THETA_MIN, TWO_THETA_MAX)
    np.savetxt(OUTPUT_FILE, table, fmt="%.6f", delimiter="\t",
               header=header, comments="")
    print("\nSaved: {0}".format(OUTPUT_FILE))

# grafici
plot_pats = sorted([p for p in patterns if np.isfinite(p[0])], key=lambda p: p[0])

if not plot_pats:
    print("Nessun diffrattogramma da disegnare (temperatura non riconosciuta nei nomi dei file?)")
else:
    temps = np.array([p[0] for p in plot_pats])
    lo = PLOT_2THETA_MIN if PLOT_2THETA_MIN is not None else min(p[1].min() for p in plot_pats)
    hi = PLOT_2THETA_MAX if PLOT_2THETA_MAX is not None else max(p[1].max() for p in plot_pats)

    try:
        cmap = plt.get_cmap(CMAP_NAME)
    except ValueError:
        cmap = plt.get_cmap("jet")
    norm = colors.Normalize(vmin=temps.min(), vmax=temps.max())
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    max_int = 0.0
    for temp, tt, inten, name in plot_pats:
        m = (tt >= lo) & (tt <= hi)
        if m.any():
            max_int = max(max_int, inten[m].max())
    step = WATERFALL_OFFSET * max_int

    fig1, ax1 = plt.subplots(figsize=(10, 6))
    for i, (temp, tt, inten, name) in enumerate(plot_pats):
        m = (tt >= lo) & (tt <= hi)
        ax1.plot(tt[m], inten[m] + i * step, color=cmap(norm(temp)), lw=0.8)
    ax1.set_xlim(lo, hi)
    ax1.set_xlabel("2theta (deg)")
    ax1.set_ylabel("Intensity (a.u.)")
    ax1.set_title("Diffractograms ({0} scans)".format(len(plot_pats)))
    fig1.colorbar(sm, ax=ax1, label="Temperature ({0})".format(TEMP_UNIT))
    fig1.tight_layout()
    fig1.savefig(OUTPUT_FILE + "_overlay.png", dpi=200)
    static_slides = [("Diffractograms", OUTPUT_FILE + "_overlay.png")]
    print("Saved: {0}".format(OUTPUT_FILE + "_overlay.png"))

    if len(plot_pats) >= 2:
        grid = np.linspace(lo, hi, 1000)
        rows = []
        for temp, tt, inten, name in plot_pats:
            order = np.argsort(tt)
            rows.append(np.interp(grid, tt[order], inten[order],
                                  left=np.nan, right=np.nan))
        Z = np.ma.masked_invalid(np.vstack(rows))

        fig2, ax2 = plt.subplots(figsize=(10, 6))
        mesh = ax2.pcolormesh(grid, temps, Z, cmap=cmap, shading="gouraud",
                              vmin=0, vmax=np.percentile(Z.compressed(), 99.5))
        fig2.colorbar(mesh, ax=ax2, label="Intensity (a.u.)")

        if SHOW_PEAK_TRACK:
            t = table[np.isfinite(table[:, 0]) & np.isfinite(table[:, 1])]
            t = t[(t[:, 1] >= lo) & (t[:, 1] <= hi)]
            if len(t):
                ax2.plot(t[:, 1], t[:, 0], color="white", lw=1.2, label="Peak position")
                ax2.legend(loc="upper right")

        ax2.set_xlim(lo, hi)
        ax2.set_xlabel("2theta (deg)")
        ax2.set_ylabel("Temperature ({0})".format(TEMP_UNIT))
        ax2.set_title("Diffractogram map")
        fig2.tight_layout()
        fig2.savefig(OUTPUT_FILE + "_heatmap.png", dpi=200)
        static_slides.append(("Diffractogram map", OUTPUT_FILE + "_heatmap.png"))
        print("Saved: {0}".format(OUTPUT_FILE + "_heatmap.png"))

    plt.show()

    # gif
    gif_done = []
    if MAKE_GIF:
        part1 = [p for p in plot_pats if p[0] <= SPLIT_TEMP]
        part2 = [p for p in plot_pats if p[0] >= SPLIT_TEMP]
        jobs = [
            (part1, OUTPUT_FILE + "_evolution_RT-{0:g}.gif".format(SPLIT_TEMP),
             "Room T to {0:g} {1}".format(SPLIT_TEMP, TEMP_UNIT), False),
            (part2, OUTPUT_FILE + "_evolution_{0:g}-end.gif".format(SPLIT_TEMP),
             "{0:g} {1} to end".format(SPLIT_TEMP, TEMP_UNIT), True),
        ]
        for pats, gif_path, tag, with_rutile in jobs:
            if len(pats) < 2:
                print("GIF non creata ({0}): meno di 2 diffrattogrammi in questo intervallo "
                      "(controlla SPLIT_TEMP)".format(os.path.basename(gif_path)))
                continue
            try:
                print("Creo {0} ...".format(os.path.basename(gif_path)))
                how, n_frames, n_pats, frames = make_gif(
                    pats, gif_path, cmap, lo, hi, peak_by_name, tag, norm,
                    build_label_specs(with_rutile) if SHOW_PEAK_LABELS else [])
                print("Saved: {0}  ({1} diffrattogrammi, {2} frames, scritta con {3})".format(
                    gif_path, n_pats, n_frames, how))
                gif_done.append((tag, gif_path))
                if MAKE_MP4:
                    mp4_path = gif_path[:-4] + ".mp4"
                    try:
                        write_mp4(mp4_path, frames, GIF_FPS)
                        print("Saved: {0}".format(mp4_path))
                    except Exception as e:
                        print("MP4 non creato ({0}): serve ffmpeg "
                              "(es. imageio.plugins.ffmpeg.download())".format(type(e).__name__))
                try:
                    display(IPyImage(filename=gif_path))
                except Exception:
                    pass
            except Exception:
                import traceback
                traceback.print_exc()
                print("GIF NON creata: vedi l'errore qui sopra.")

    # powerpoint
    if MAKE_PPTX:
        pptx_path = OUTPUT_FILE + "_presentation.pptx"
        try:
            build_pptx(pptx_path, static_slides + gif_done)
            print("Saved: {0}  ({1} slide; le GIF si animano in modalita' presentazione)".format(
                pptx_path, len(static_slides) + len(gif_done)))
        except ImportError:
            print("python-pptx non installato: per la presentazione esegui  "
                  "pip install \"python-pptx<1\"  e riavvia il kernel")
        except Exception:
            import traceback
            traceback.print_exc()
            print("PPTX NON creato: vedi l'errore qui sopra.")